# Safety Score & OCR
# Alur Verifikasi Dokumen Ayomi

**Peran fitur di aplikasi**
Fitur ini membantu orang tua melihat kelengkapan pemeriksaan dokumen legalitas daycare. Pengelola daycare mengunggah dokumen melalui dashboard; OCR membaca isinya; sistem memeriksa data yang dapat dibaca dan menandai hal yang perlu ditinjau admin. Setelah admin mengonfirmasi, Ayomi menampilkan status/badge di profil daycare.

Safety Score di prototipe ini berarti skor kelengkapan verifikasi dokumen. Skor bukan penilaian keselamatan operasional, bukan sertifikasi, dan tidak menyatakan dokumen sah secara hukum. Dokumen yang berhasil terbaca OCR tetap perlu pemeriksaan manusia.



# 1. Alur pengguna dan sistem
1. Pengelola daycare mengunggah dokumen pada bagian verifikasi profil Ayomi, memilih jenis dokumen dari daftar yang ditetapkan tim, serta mengisi data profil dasar daycare.
2. Sistem memvalidasi file (jenis, ukuran, jumlah halaman) dan menyimpan file secara privat. Status awal: "Menunggu diproses".
3. OCR membaca teks tiap halaman. Teks hasil ekstraksi dan halaman sumber disimpan untuk audit; hasil OCR rendah/tidak terbaca langsung masuk "Perlu tinjauan manual".
4. Ekstraksi terstruktur mengambil kandidat nama daycare/pengelola, nomor dokumen, instansi penerbit, alamat, tanggal terbit, dan tanggal berlaku "sesuai field" untuk tipe dokumen itu.
5. Pemeriksaan otomatis membandingkan kandidat dengan profil daycare dan aturan checklist: field wajib ada, nama/alamat konsisten, tanggal masih berlaku jika dokumen punya masa berlaku, dan tidak ada konflik antarberkas. OCR tidak membuktikan keaslian.
6. Admin Ayomi meninjau file asli, hasil OCR, dan flag sistem; admin menyetujui, menolak dengan alasan, atau meminta unggahan ulang. Semua keputusan dicatat.
7. Ayomi menampilkan hasil di halaman daycare: "Dokumen terverifikasi", "Sebagian dokumen terverifikasi", "Perlu dilengkapi", atau "Dalam peninjauan", disertai waktu pembaruan dan jenis dokumen yang sudah diperiksa.
Status yang disarankan
- Dalam peninjauan: OCR selesai, menunggu pemeriksaan admin.
- Perlu dilengkapi: ada berkas/field wajib yang hilang atau perlu unggahan ulang.
- Terverifikasi: admin sudah memeriksa dan menyetujui seluruh dokumen wajib.
- Perlu diperbarui: dokumen yang memiliki tanggal berlaku sudah kedaluwarsa atau mendekati kedaluwarsa.
- Tidak terverifikasi: admin menolak dokumen dengan alasan yang tercatat.
Badge publik sebaiknya memakai status verifikasi yang disetujui admin. Jangan memberi badge "aman" hanya dari nilai OCR/skor otomatis.

# 2. Data yang diproses
Data profil daycare
daycare_id, nama resmi daycare, alamat, nama pengelola (bila relevan), dan daftar dokumen wajib`.
Data berkas
Berkas PDF/gambar, tipe dokumen pilihan pengguna, waktu unggah, uploader, status review, dan versi berkas. Dokumen asli harus disimpan privat; notebook demo tidak menyimpan ke database.
Field OCR per dokumen
Field wajib ditentukan berdasarkan jenis dokumen. Contoh field generik untuk eksperimen: nama lembaga, nomor dokumen, penerbit, alamat, tanggal terbit, tanggal akhir berlaku. Field yang tidak relevan boleh ditandai "tidak berlaku", bukan dianggap gagal.
Jenis dokumen wajib dan aturan validitas harus ditetapkan Ayomi/admin dengan merujuk ketentuan yang berlaku. Daftar di notebook ini tidak menetapkan standar hukum.

In [ ]:
# @title 3. Setup OCR di Google Colab
!pip -q install pytesseract pdf2image pillow ipywidgets
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-ind poppler-utils
import re, os, tempfile
from pathlib import Path
from datetime import date, datetime
import pandas as pd, pytesseract, ipywidgets as widgets
from PIL import Image, ImageOps, ImageEnhance
from pdf2image import convert_from_path
from IPython.display import display, HTML
print("Setup OCR siap")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Setup OCR siap


# 4. Model pemeriksaan dan skor prototipe
Skor mengukur kelengkapan bukti dokumen dalam checklist demo, bukan tingkat keamanan daycare. Supaya masuk akal untuk aplikasi: tiap dokumen wajib memiliki bobot; indikator yang terbaca lengkap mendapat nilai penuh, sebagian/meragukan mendapat setengah dan flag review, tidak ditemukan mendapat nol. Dokumen kedaluwarsa mendapat nol untuk indikator masa berlaku dan flag admin. Jika OCR gagal, jangan hitung skor.

Status verifikasi tetap ditentukan admin. Sebelum review, UI hanya menampilkan "Dalam peninjauan" atau "Perlu dilengkapi". Bobot dan contoh indikator di bawah hanya untuk demonstrasi alur, wajib diganti checklist yang disetujui Ayomi.

In [ ]:
# @title 5. Checklist demo dan fungsi status
# Contoh struktur; bukan daftar legal resmi. Sesuaikan per tipe dokumen.
CHECKLIST_DEMO={
 "Dokumen contoh A":{"bobot":50,"field_wajib":["nama_lembaga","nomor_dokumen","penerbit"]},
 "Dokumen contoh B":{"bobot":30,"field_wajib":["nama_lembaga","alamat"]},
 "Dokumen contoh C":{"bobot":20,"field_wajib":["nama_lembaga","tanggal_berlaku"]},
}

def status_field(value):
 if value is None or not str(value).strip(): return "Tidak ditemukan"
 if str(value).strip().upper() in {"PERLU TINJAUAN","TIDAK TERBACA"}: return "Perlu tinjauan"
 return "Terbaca ? perlu konfirmasi"

def hitung_skor_demo(hasil_per_dokumen):
 """hasil_per_dokumen: {jenis: {field: nilai_ocr}}; nilai tidak berarti sah."""
 total=0; maksimum=sum(x["bobot"] for x in CHECKLIST_DEMO.values()); rows=[]; perlu_review=[]
 for jenis,spec in CHECKLIST_DEMO.items():
  data=hasil_per_dokumen.get(jenis,{})
  ada=sum(bool(data.get(f)) and data.get(f)!="TIDAK TERBACA" for f in spec["field_wajib"])
  bobot=spec["bobot"]; poin=bobot*ada/len(spec["field_wajib"]); total+=poin
  rows.append({"Dokumen":jenis,"Field terisi":f"{ada}/{len(spec['field_wajib'])}","Poin kelengkapan":round(poin,1)})
  if ada<len(spec["field_wajib"]): perlu_review.append(jenis)
 return {"skor":round(100*total/maksimum) if maksimum else 0,"rows":rows,"dokumen_perlu_review":perlu_review,"status_ui":"Dalam peninjauan"}

# Demo saja: sistem tidak menyetujui dokumen; admin tetap menentukan status final.

# 6. OCR dan ekstraksi teks
Tahap ini hanya mengubah gambar/PDF menjadi teks. Untuk MVP, tampilkan teks OCR kepada admin dan minta konfirmasi field penting, karena ekstraksi nama/nomor/tanggal otomatis bisa keliru. Setelah contoh dokumen tersedia, pola ekstraksi dapat dibuat khusus untuk tiap tipe dokumen.

In [ ]:
# @title 7. Fungsi OCR PDF / gambar
IMAGE_EXT={".png",".jpg",".jpeg",".tif",".tiff",".bmp",".webp"}
def preprocess(im):
 im=ImageOps.exif_transpose(im).convert("L")
 return ImageEnhance.Contrast(im).enhance(1.6)
def baca_ocr(path):
 path=Path(path); pages=[]
 if path.suffix.lower()==".pdf":
  pages=convert_from_path(str(path),dpi=250)
 elif path.suffix.lower() in IMAGE_EXT:
  pages=[Image.open(path)]
 else: raise ValueError("Gunakan PDF atau gambar PNG/JPG/TIFF/BMP/WEBP")
 return [{"halaman":i+1,"teks":pytesseract.image_to_string(preprocess(im),lang="ind+eng")} for i,(im,p) in enumerate(zip(pages,pages))]

# 7. Simulasi dashboard pemeriksaan
Notebook ini mendemonstrasikan unggah dan OCR. Dalam aplikasi Ayomi, komponen ini menjadi halaman pengelola untuk mengunggah dokumen, dan halaman admin untuk review. Jangan tampilkan isi dokumen sensitif ke orang tua; tampilkan status dan tanggal verifikasi saja.

In [ ]:
# @title 8. Upload dan tampilkan OCR untuk review
upload=widgets.FileUpload(accept=".pdf,.png,.jpg,.jpeg,.tif,.tiff,.bmp,.webp",multiple=False)
btn=widgets.Button(description="Jalankan OCR",button_style="primary"); out=widgets.Output()
def proses(_):
 with out:
  out.clear_output()
  if not upload.value: print("Pilih file terlebih dahulu."); return
  obj=upload.value
  if isinstance(obj,dict): filename,meta=next(iter(obj.items())); content=meta["content"]
  else: meta=obj[0]; filename=meta["name"]; content=meta["content"]
  path=None
  try:
   with tempfile.NamedTemporaryFile(delete=False,suffix=Path(filename).suffix) as f: f.write(bytes(content)); path=f.name
   pages=baca_ocr(path); teks="\n".join(x["teks"] for x in pages)
   print("Berkas:",filename,"| Halaman:",len(pages),"| Status: Dalam peninjauan")
   if len(teks.strip())<30: print("OCR tidak memadai ? minta unggah ulang atau input manual.")
   print("\n--- Teks OCR untuk dicocokkan admin dengan dokumen asli ---\n")
   print(teks[:12000] or "Tidak ada teks terbaca")
  except Exception as e: print(type(e).__name__+":",e)
  finally:
   if path and os.path.exists(path): os.remove(path)
btn.on_click(proses); display(widgets.VBox([upload,btn,out]))

# 8. Evaluasi dan integrasi ke Ayomi
Uji prototipe
Gunakan berkas contoh per tipe: scan jelas, buram, PDF multi-halaman, field hilang, serta tanggal lewat. Ukur akurasi OCR untuk nama/nomor/tanggal dan berapa banyak yang harus ditinjau manual. Jangan menilai kualitas legal hanya dari akurasi OCR.
Bentuk integrasi produksi yang disarankan
- Pengelola: unggah/lihat status/melengkapi berkas.
- OCR service: ekstrak teks dan kandidat field, simpan versi hasil serta confidence.
- Admin Ayomi: review berkas dan hasil ekstraksi; approve/reject/request update dengan catatan.
- Database: simpan metadata, akses berkas privat, riwayat status dan reviewer; batasi akses berbasis peran.
- UI orang tua: tampilkan badge/status terverifikasi, jenis pemeriksaan dan tanggal; jangan tampilkan dokumen mentah.
- Pengingat: tandai dokumen yang mendekati tanggal berlaku untuk pembaruan.
Batasan penting: OCR bukan pemeriksaan keaslian; skor adalah kelengkapan checklist; admin harus memeriksa file asli sebelum status verifikasi tampil.